In [ ]:
!pip install captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from transformers import AdamW
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import os
from captum.attr import visualization as viz
from captum.attr import LayerConductance, LayerIntegratedGradients
from captum.attr import Saliency, visualization
from captum.attr import TokenReferenceBase
import shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df1 = pd.read_csv('../data/letters_2021_processed.csv')
df1 = df1[['s1_s2', 'full_text', 'LETTER_GENDER']]
df1 = df1.rename(columns={'LETTER_GENDER':'label'})

In [ ]:
df2 = pd.read_csv('../data/sentence_sets_trimmed_processed.csv')
df2 = df2[['s1_s2', 'full_text', 'applicant_gender']]
df2 = df2.rename(columns={'applicant_gender':'label'})

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)

In [ ]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [ ]:
df['label'] = df['label'].replace(gender_label_mapping)

<ipython-input-179-cc45885305bc>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace(gender_label_mapping)


# Create Training and Test Sets

In [ ]:
train_text, temp_text, train_labels, temp_labels = train_test_split(df['s1_s2'], df['label'],
                                                                    random_state=0,
                                                                    test_size=0.3,
                                                                    stratify=df['label'])


val_text, test_text, val_labels, test_labels = train_test_split(temp_text, temp_labels,
                                                                random_state=0,
                                                                test_size=0.5,
                                                                stratify=temp_labels)

In [ ]:
bert = AutoModel.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [ ]:
tokens_train = tokenizer.batch_encode_plus(
    train_text.tolist(),
    padding='max_length',
    truncation=True
)

tokens_val = tokenizer.batch_encode_plus(
    val_text.tolist(),
    padding='max_length',
    truncation=True
)

tokens_test = tokenizer.batch_encode_plus(
    test_text.tolist(),
    padding='max_length',
    truncation=True
)

In [ ]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels.tolist())

In [ ]:
batch_size = 8
train_data = TensorDataset(train_seq, train_mask, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_seq, val_mask, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [ ]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


# Create Model

In [ ]:
class BERT_Arch(nn.Module):

    def __init__(self, bert):

        super(BERT_Arch, self).__init__()

        self.bert = bert

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu =  nn.ReLU()

        # dense layer 1
        self.fc1 = nn.Linear(768,512)

        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(512,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

        #pass the inputs to the model
        distilbert_output = self.bert(sent_id, attention_mask=mask)

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]
        # pooled_output = hidden_state.mean(dim=1)

        x = self.fc1(pooled_output)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)

        # apply softmax activation
        x = self.softmax(x)

        return x

In [ ]:
model = BERT_Arch(bert)
model = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss(weight=weights)
epochs = 10

In [ ]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_dataloader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    # push the batch to gpu
    batch = [r.to(device) for r in batch]

    sent_id, mask, labels = batch

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(sent_id, mask)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_dataloader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: ', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: ', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [ ]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_dataloader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    # push the batch to gpu
    batch = [t.to(device) for t in batch]

    sent_id, mask, labels = batch

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(sent_id, mask)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_dataloader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: ', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: ', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, total_preds

In [ ]:
# set initial loss to infinite
best_valid_loss = float('inf')

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, _ = evaluate()

    #save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print('Model Saved!')
        torch.save(model, '../saved_models/saved_model.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/10 [00:00<?, ?it/s]


 Epoch 1 / 10


Training:   0%|          | 0/787 [00:00<?, ?it/s]

Training Classification Report:                precision    recall  f1-score   support

           0       0.35      0.41      0.38      1950
           1       0.71      0.65      0.68      4340

    accuracy                           0.58      6290
   macro avg       0.53      0.53      0.53      6290
weighted avg       0.60      0.58      0.59      6290

Training Confusion Matrix:  [[ 806 1144]
 [1510 2830]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       0.49      0.12      0.20       418
           1       0.71      0.94      0.81       930

    accuracy                           0.69      1348
   macro avg       0.60      0.53      0.50      1348
weighted avg       0.64      0.69      0.62      1348

Validation Confusion Matrix:  [[ 52 366]
 [ 54 876]]
Model Saved!

Training Loss: 0.689
Validation Loss: 0.683

 Epoch 2 / 10


Training:   0%|          | 0/787 [00:00<?, ?it/s]

...

Training Classification Report:                precision    recall  f1-score   support

           0       0.49      0.64      0.55      1950
           1       0.81      0.70      0.75      4340

    accuracy                           0.68      6290
   macro avg       0.65      0.67      0.65      6290
weighted avg       0.71      0.68      0.69      6290

Training Confusion Matrix:  [[1246  704]
 [1310 3030]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       0.61      0.36      0.45       418
           1       0.76      0.89      0.82       930

    accuracy                           0.73      1348
   macro avg       0.68      0.63      0.64      1348
weighted avg       0.71      0.73      0.71      1348

Validation Confusion Matrix:  [[152 266]
 [ 99 831]]
Model Saved!

Training Loss: 0.610
Validation Loss: 0.617

 Epoch 9 / 10


Training:   0%|          | 0/787 [00:00<?, ?it/s]

Training Classification Report:                precision    recall  f1-score   support

           0       0.49      0.62      0.55      1950
           1       0.81      0.71      0.76      4340

    accuracy                           0.69      6290
   macro avg       0.65      0.67      0.65      6290
weighted avg       0.71      0.69      0.69      6290

Training Confusion Matrix:  [[1217  733]
 [1246 3094]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       0.56      0.50      0.53       418
           1       0.79      0.83      0.81       930

    accuracy                           0.72      1348
   macro avg       0.67      0.66      0.67      1348
weighted avg       0.72      0.72      0.72      1348

Validation Confusion Matrix:  [[209 209]
 [162 768]]
Model Saved!

Training Loss: 0.598
Validation Loss: 0.596

 Epoch 10 / 10


Training:   0%|          | 0/787 [00:00<?, ?it/s]

Training Classification Report:                precision    recall  f1-score   support

           0       0.50      0.66      0.57      1950
           1       0.82      0.70      0.75      4340

    accuracy                           0.69      6290
   macro avg       0.66      0.68      0.66      6290
weighted avg       0.72      0.69      0.70      6290

Training Confusion Matrix:  [[1289  661]
 [1315 3025]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       0.55      0.57      0.56       418
           1       0.80      0.79      0.80       930

    accuracy                           0.72      1348
   macro avg       0.68      0.68      0.68      1348
weighted avg       0.72      0.72      0.72      1348

Validation Confusion Matrix:  [[237 181]
 [195 735]]
Model Saved!

Training Loss: 0.591
Validation Loss: 0.585


In [ ]:
# model.load_state_dict(torch.load('../saved_models/saved_weights.pt'))

In [ ]:
def forward_func(input_ids, mask, target_class):
    output = model(input_ids, mask=mask)
    return output[:, target_class]


In [ ]:
lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

In [ ]:
def add_attributions_to_visualizer(attributions, input_ids, pred, pred_ind, label, delta, vis_data_records):
    # Sum attributions across embedding dimensions

    attributions = attributions.sum(dim=2).squeeze(0)

    # Normalize attributions
    attributions = attributions / torch.norm(attributions)

    # Convert to numpy for visualizations
    attributions = attributions.cpu().detach().numpy()

    # Convert token IDs to words
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())

    # Get predicted and actual labels
    pred_label = label_map.get(pred_ind, "Unknown")
    actual_label = label_map.get(label, "Unknown")

    # Store visualization data
    vis_data_records.append(visualization.VisualizationDataRecord(
                            attributions,
                            pred,  # Raw prediction score
                            pred_label,
                            actual_label,
                            "Target Label",
                            attributions.sum(),
                            tokens,
                            delta))

    return vis_data_records  # Return updated list for visualization

In [ ]:
# Get padding token index from the tokenizer
PAD_IND = tokenizer.pad_token_id

# Initialize TokenReferenceBase with padding token index
token_reference = TokenReferenceBase(reference_token_idx=PAD_IND)

vis_data_records_ig = []

label_map = {0: "Female", 1: "Male"}

In [ ]:
def interpret_sentence(model, sentence):
    # text = [tok.text for tok in nlp.tokenizer(sentence.lower())]

    device = torch.device("cpu")
    model.to(device)

    target_class = 1

    encoded = tokenizer.batch_encode_plus(
        [sentence],
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    # if len(text) < min_len:
    #     text += [TEXT.pad_token] * (min_len - len(text))
    # indexed = [TEXT.vocab.stoi[t] for t in text]

    input_indices = encoded['input_ids'].to(device)  # Extract input IDs
    attention_mask = encoded['attention_mask'].to(device)  # Extract mask
    seq_length=(attention_mask.squeeze() == 1).sum().item()

    input_indices = input_indices[:, :seq_length]  # Keep only non-padding tokens
    attention_mask = attention_mask[:, :seq_length]  # Match the shape

    model.zero_grad()

    pred = forward_func(input_indices, attention_mask, target_class).item()
    pred_ind = round(pred)

    # generate reference indices for each sample
    reference_indices = token_reference.generate_reference(seq_length, device=device).unsqueeze(0)

    # compute attributions and approximation delta using layer integrated gradients
    attributions_ig, delta = lig.attribute(inputs=input_indices, additional_forward_args=(attention_mask, target_class),
                                           return_convergence_delta=True)
    print(attributions_ig)


    add_attributions_to_visualizer(attributions_ig, input_indices, pred, pred_ind, target_class, delta, vis_data_records_ig)

In [ ]:
idx = 10
interpret_sentence(model, test_text.iloc[idx])

tensor([[[ 3.8209e-04, -4.4233e-03,  5.3174e-04,  ..., -2.4034e-04,
          -8.3456e-04, -3.1569e-04],
         [ 2.7310e-05,  1.6885e-06,  6.0554e-06,  ..., -8.9220e-06,
           5.3706e-05,  9.7310e-05],
         [-1.4735e-04, -6.2999e-05, -2.0413e-04,  ...,  5.5798e-05,
          -6.3748e-06,  1.2090e-04],
         ...,
         [-3.7409e-04, -2.0508e-04, -2.6625e-04,  ..., -7.7084e-05,
          -3.4276e-05,  1.1018e-03],
         [-6.8468e-05,  4.4831e-04, -2.6483e-06,  ...,  3.4118e-05,
           5.4170e-04,  2.3986e-05],
         [-4.3549e-04,  8.4649e-04,  4.1463e-04,  ..., -1.2672e-04,
          -4.4430e-04, -9.3499e-05]]], dtype=torch.float64)


In [ ]:
print('Visualize attributions based on Integrated Gradients')
_ = visualization.visualize_text(vis_data_records_ig)

Visualize attributions based on Integrated Gradients


# Shap values

In [ ]:
# def f(x):
#     encoded_inputs = [tokenizer.encode_plus(v, padding="max_length", truncation=True, return_tensors="pt") for v in x]

#     # Unpacking the encoded inputs into input_ids and attention_masks
#     input_ids = torch.cat([e['input_ids'] for e in encoded_inputs], dim=0).to(device)
#     attention_masks = torch.cat([e['attention_mask'] for e in encoded_inputs], dim=0).to(device)

#     output = model(input_ids, attention_masks).detach().cpu().numpy()

#     return output[:, 1]

In [ ]:
# explainer = shap.Explainer(f, tokenizer)

In [ ]:
# df

In [ ]:
# #observations = df['s1_s2'][8970:]
# observations = df['s1_s2'][:100]
# shap_values = explainer(observations, fixed_context=1)

In [ ]:
# shap.plots.text(shap_values[68])

In [ ]:
# shap.plots.bar(shap_values.mean(0), order=shap.Explanation.argsort, max_display=15)

In [ ]:
# shap.plots.bar(shap_values.mean(0), order=shap.Explanation.argsort[::-1], max_display=15)

In [ ]:
# shap.plots.bar(shap_values.mean(0), order=shap.Explanation.argsort[::-1], max_display=10)

In [ ]:
# shap.plots.bar(shap_values.mean(0), order=shap.Explanation.argsort, max_display=10)

In [ ]:
# shap.plots.bar(shap_values.sum(0), order=shap.Explanation.argsort, max_display=50)

In [ ]:
# shap.plots.bar(shap_values.sum(0), order=shap.Explanation.argsort[::-1], max_display=50)

In [ ]:
# text = df['s1_s2'][6]

In [ ]:
#tokenizer.encode_plus(text, padding="max_length", truncation=True, return_tensors="pt")

In [ ]:
# tokenizer.tokenize(text)